# 4 · Counting: `_complete_counts` and `_init_cpts`

**Trusts:** notebook 1, and `CPT.from_counts`.

With nothing missing, learning a CPT is counting: P(CO = c | hour = h) ≈ (rows with hour h and CO c) / (rows with hour h). `_complete_counts` does that for every node using only the rows where the whole family is observed. `_init_cpts` turns those counts into EM's starting point.

In [ ]:
%load_ext autoreload
%autoreload 2

import itertools

import numpy as np
import pandas as pd
from toy import (
    CARDS,
    CO,
    DATA,
    EVIDENCE,
    NO2,
    S1,
    S4,
    STRUCTURE,
    TABLES,
    H,
    T,
    cpts,
    network,
    same,
    toy_bn,
)

from cse250a.modules.core import CPT

net = network()
bn = toy_bn()   # DiscreteBN with the hand-picked CPTs, not fitted

## Step 1: one node, a plain loop

Count (hour, CO) pairs in the toy data, skipping rows where either is missing:

In [ ]:
counts = np.zeros((2, 2))   # axes: hour, CO
for _, row in DATA.iterrows():
    if pd.isna(row[H]) or pd.isna(row[CO]):
        continue
    counts[int(row[H]), int(row[CO])] += 1

print(pd.DataFrame(counts, index=["hour=0", "hour=1"], columns=["CO=0", "CO=1"]))
print("rows used:", int(counts.sum()), "of", len(DATA))

## Step 2: the same count without a Python loop, and the `np.add.at` trap

The fast way: select the family's columns, keep complete rows, and use them as indices. The obvious `counts[idx] += 1` **is wrong**: when an index repeats, numpy writes the cell once, not once per repeat.

In [ ]:
idx = (np.array([0, 0, 1]), np.array([1, 1, 0]))   # cell (0, 1) appears twice

wrong = np.zeros((2, 2)); wrong[idx] += 1
right = np.zeros((2, 2)); np.add.at(right, idx, 1)
print("counts[idx] += 1 →\n", wrong, "\nnp.add.at       →\n", right)

In [ ]:
rows = DATA[[H, CO]].to_numpy()
complete = ~np.isnan(rows).any(axis=1)
step2 = np.zeros((2, 2))
np.add.at(step2, tuple(rows[complete].astype(int).T), 1)
same("vectorized = loop", step2, counts)

`rows[complete].astype(int).T` has one row per family column, so `tuple(...)` gives one index array per table axis: exactly what `np.add.at` wants.

## Step 3: any node, every node

Swap the hard-coded `[H, CO]` for `net.family(node)` and the `(2, 2)` for `net.shape(node)`. S1's family has three columns (CO, T, S1), so its counts are a 2×2×2 array. The code doesn't change.

In [ ]:
def complete_counts(data):
    out = {}
    for node in net.structure:
        cols = list(net.family(node))
        rows = data[cols].to_numpy()
        keep = rows[~np.isnan(rows).any(axis=1)].astype(int)
        out[node] = np.zeros(net.shape(node))
        np.add.at(out[node], tuple(keep.T), 1)
    return out

mine = complete_counts(DATA)
for node in net.nodes:
    print(f"{node!s:>14}: {int(mine[node].sum())} complete rows")
same("all nodes", mine, bn._complete_counts(DATA))

## Step 4: latent families get zeros

Add a latent **Event** above both gases (Phase 3b). It has no column, so **no row** is complete for any family that contains it. Those counts are all zeros, and EM has to fill them in. In `bn.py` this is the `if any(c not in data.columns ...)` branch.

```
  hour ──► Event ──► CO ──► S1 ◄── T
                 └─► NO2 ─► S4 (◄── CO)
```

In [ ]:
from cse250a.modules.core import Network
from cse250a.modules.models import DiscreteBN

EVENT = "Event"
latent_net = Network(
    {EVENT: (H,), CO: (EVENT,), NO2: (EVENT,), S1: (CO, T), S4: (NO2, CO)},
    {**CARDS, EVENT: 2},
    latent=(EVENT,),
).validate(DATA.columns)
latent_bn = DiscreteBN(latent_net)

lc = latent_bn._complete_counts(DATA)
for node in latent_net.nodes:
    print(f"{node!s:>14}  family {tuple(map(str, latent_net.family(node)))!s:40} rows counted: {int(lc[node].sum())}")

## Step 5: counts → CPT, with smoothing

`CPT.from_counts` adds a pseudo-count α to every cell, then normalizes each parent combination:

$$P(\text{node}=j \mid \text{parents}) = \frac{n_j + \alpha}{N + k\alpha}$$

By hand for P(CO | hour) from the step 1 counts:

In [ ]:
for alpha in (0.0, 1.0):
    by_hand = (counts + alpha) / (counts + alpha).sum(axis=1, keepdims=True)
    same(f"alpha={alpha}", by_hand, CPT.from_counts(CO, [H], counts, alpha).table)
    print(by_hand.round(3))

With α = 0 a cell never seen gets probability 0, and later `log 0 = -inf`. α = 1 (Laplace) keeps every cell positive. With little data it pulls the CPT noticeably toward uniform. With a lot of data it barely matters.

## Step 6: `_init_cpts`, EM's starting point

Three rules, in order:

1. **Family has no latent member:** `from_counts(complete counts, α)`. With nothing missing this is already the final answer.
2. **Family touches a latent node:** random rows from a Dirichlet, seeded. *Random, not uniform*: if every latent state started identical, the E-step would treat them identically forever and they'd never separate.
3. **`init` given for the node:** use that table instead (shape-checked).

In [ ]:
def init_cpts(model, data):
    rng = np.random.default_rng(model.seed)
    complete = model._complete_counts(data)
    out = {}
    for node, parents in model.network.structure.items():
        shape = model.network.shape(node)
        if any(c in model.network.latent for c in model.network.family(node)):
            rows = rng.dirichlet(np.ones(shape[-1]), size=int(np.prod(shape[:-1])))
            table = rows.reshape(shape)
        else:
            table = CPT.from_counts(node, parents, complete[node], model.alpha).table
        if node in model.init:
            table = np.asarray(model.init[node], dtype=float)
        out[node] = table
    return out

for name, model in [("static", DiscreteBN(net)), ("latent", latent_bn),
                    ("latent + init", DiscreteBN(latent_net, init={EVENT: np.full((2, 2), 0.5)}))]:
    theirs = {n: c.table for n, c in model._init_cpts(DATA).items()}
    same(name, init_cpts(model, DATA), theirs)

For the latent case to match, the random draws must happen **in the same order**: one `rng.dirichlet` call per latent family, in `structure` order, from a generator seeded with `model.seed`. Same seed → same start → reproducible EM.